# 🦀 Day 5: Derive Macros with syn and quote
---

Today we build **derive macros** using `syn` and `quote`.

- **syn**: Parse Rust code into AST (`DeriveInput`)
- **quote**: Generate Rust code from templates
- **DeriveInput**: name, generics, data (struct fields)
- **Parsing fields**: Named vs unnamed (tuple structs)
- **quote!**: `#variable` interpolation, `#(#iter)*` repetition
- **#[derive(Describe)]**: Generate `describe()` listing all fields

In [ ]:
:dep syn = { version = "2", features = ["full"] }
:dep quote = "1"
:dep proc-macro2 = "1"

In [ ]:
// DeriveInput structure (conceptual)
// pub struct DeriveInput {
//     pub attrs: Vec<Attribute>,
//     pub vis: Visibility,
//     pub ident: Ident,        // struct name
//     pub generics: Generics,
//     pub data: Data,          // DataStruct { fields }, DataEnum { variants }, DataUnion
// }
//
// DataStruct has fields: Fields::Named (NamedField { name, ty }) or Fields::Unnamed

use quote::{quote, format_ident};

// quote! interpolation: #name expands the variable
let name = format_ident!("Foo");
let tokens = quote! {
    struct #name {
        x: i32,
    }
};
println!("quote output: {}", tokens);

In [ ]:
// quote! repetition: #(#items)* or #(#items),*
let items = vec![quote! { 1 }, quote! { 2 }, quote! { 3 }];
let sum = quote! {
    let arr = [#(#items),*];
    arr.iter().sum::<i32>()
};
println!("Repetition: {}", sum);

## Full #[derive(Describe)] Implementation

In a proc-macro crate (`my_macros`), the implementation would look like:

```rust
#[proc_macro_derive(Describe)]
pub fn describe_derive(input: TokenStream) -> TokenStream {
    let ast = syn::parse_macro_input!(input as syn::DeriveInput);
    let name = &ast.ident;
    let fields = match &ast.data {
        syn::Data::Struct(s) => match &s.fields {
            syn::Fields::Named(n) => &n.named,
            _ => unimplemented!(),
        },
        _ => unimplemented!(),
    };
    let field_names = fields.iter().map(|f| &f.ident);
    let field_tys = fields.iter().map(|f| &f.ty);
    quote! {
        impl #name {
            pub fn describe(&self) -> String {
                format!(
                    "{}: {} = {:?}, {} = {:?}",
                    stringify!(#(#field_names),*),
                    stringify!(#(#field_tys),*),
                    self.#(#field_names),*,
                    // ... (simplified)
                )
            }
        }
    }.into()
}
```

In [ ]:
// Testing derive macros: create a Cargo project, add dependency, use #[derive(Describe)]
// cargo new describe_demo && cd describe_demo
// Add to Cargo.toml: my_macros = { path = "../my_macros" }
// In main.rs:
// use my_macros::Describe;
// #[derive(Describe)]
// struct Person { name: String, age: u32 }
// fn main() { println!("{}", Person { name: "Alice".into(), age: 30 }.describe()); }

println!("Derive macros are tested in a separate Cargo project.");

## 📝 Exercise: Design #[derive(Builder)]

Design a `#[derive(Builder)]` macro that generates a builder for a struct.

```rust
#[derive(Builder)]
struct User { name: String, age: u32 }
// Generates: UserBuilder with .name(s).age(n).build() -> User
```

What would you iterate over? What would each `quote!` block generate?

In [ ]:
// YOUR CODE HERE — design the Builder derive:
// - For each field, generate a setter method
// - build() constructs the struct, returning Result<User, String> if fields missing

## 🎯 Key Takeaways

1. **syn** parses `TokenStream` into `DeriveInput` (ident, generics, data)
2. **quote** generates code: `#var` interpolates, `#(#iter)*` repeats
3. **Data::Struct** has `Fields::Named` (named fields) or `Fields::Unnamed` (tuple)
4. Iterate `fields.named` to get `ident` and `ty` for each field
5. Derive macros are tested in a **separate** crate that depends on the proc-macro crate
6. **Builder** pattern is a classic derive macro — generate setters and `build()`

---
**Tomorrow:** Attribute macros